# Why high-frequency covariance data must be cleaned before it is synchronized

This notebook is the pedagogical case study for the `covharness` measurement layer. It uses the WRDS millisecond TAQ sample for 13 February 2009 and five names. IBM, AAPL, MSFT, JPM, and XOM, from `taqmsamp_all.nbbom_20090213`.

The production cleaner is `covharness.data.quotes.clean_nbbo_quotes`. This notebook does not re-implement Q1–Q4. The runnable validation that produced the committed diagnostics is `experiments/taq_five_stock_pilot.py`.

Pipeline

$$
\text{raw NBBO}
\to
\text{quote cleaning}
\to
\text{midquote}
\to
\text{previous-tick synchronization}
\to
\text{log returns}
\to
\text{realized covariance}.
$$

Previous-tick synchronization cannot repair a dirty quote. It carries the last supplied price onto the grid. Because realized covariance is $\sum_j r_j r_j^{\top}$, one artificial return contaminates an entire row and column. The cleaned matrix remains a realized-covariance proxy. It is not ground truth and not a realized-kernel estimator.


## Methodological source

Cleaning is adapted from Barndorff-Nielsen, Hansen, Lunde, and Shephard (2011), Section 5.1, following Barndorff-Nielsen, Hansen, Lunde, and Shephard (2009), *Realised kernels in practice: trades and quotes*.

P3 in the paper retains a selected single exchange. We use consolidated NBBO fields `best_bid` and `best_ask`, so the procedure is an adaptation rather than an exact replication of that exchange-selection rule.

Q4 is a symmetric ex-post filter. It flags an isolated midquote that is extreme relative to 50 neighbors. It is not designed to remove a burst of similar displaced quotes. The JPM sample below shows that distinction.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from covharness.data.quotes import clean_nbbo_quotes

RESULTS = ROOT / "results" / "taq_five_stock_pilot_20090213.json"
report = json.loads(RESULTS.read_text())
print("Loaded derived diagnostics from", RESULTS.relative_to(ROOT))
print("Sample", report["sample"])

## Optional live WRDS rerun

Authentication belongs outside the repository (for example `~/.pgpass`). Do not place a WRDS password here. Raw ticks are not written to disk. Uncomment the next cell only to recompute the diagnostics from WRDS.


In [ ]:
# import wrds
# from experiments.taq_five_stock_pilot import fetch_nbbo_sample, run_pilot
#
# db = wrds.Connection()
# try:
#     raw = fetch_nbbo_sample(db)
# finally:
#     db.close()
# report = run_pilot(raw)
# print("Recomputed live. Production cleaner:", clean_nbbo_quotes.__module__)

## Production cleaning diagnostics

Session hours were restricted in SQL to 09:30–16:00, so P1 and P2 remove nothing in this extract. Q1 dominates the row reduction because many NBBO messages share a timestamp. Q2–Q4 remove a much smaller share. Q1 is a collapse of simultaneous quotes to the median bid and ask, not a deletion of economically distinct observations.


In [ ]:
cleaning = pd.Series(report["cleaning"], name="full_panel")
by_stock = pd.DataFrame(report["cleaning_by_stock"]).T
display(cleaning.to_frame())
display(by_stock[["n_input", "n_after_q1", "n_after_q2", "n_after_q3", "n_after_q4",
                 "n_removed_q1", "n_removed_q2", "n_removed_q3", "n_removed_q4"]])

## Naive versus production-cleaned JPM five-minute midquotes

The naive panel applies only positive bid/ask, nonnegative spread, and last-quote-at-timestamp (so previous-tick sync can run). That last-quote collapse is not Q1.

The cleaned panel uses `clean_nbbo_quotes` and then `previous_tick_sync`. The 09:30 grid point is left missing if no regular-session quote exists at or before 09:30. It is not filled from pre-market data.


In [ ]:
jpm = pd.DataFrame(report["jpm_five_minute_midquotes"])
display(jpm)
print("Missing prices at 09:30 by stock")
print(report["missing_prices"])
print("Open convention")
print(report["open_convention"])

### What the production path did and did not remove

On this sample the isolated 10:05 JPM print near 22.62 is replaced by a previous-tick midquote near 25.24 after Q1–Q4. The 09:45 grid time remains near 22.585 on both the naive and cleaned panels.

That is not a failure of previous-tick matching. Immediately before 09:45 the NBBO stream contains a burst of displaced quotes near 22.5–22.6 interleaved with quotes near 25. Q4 looks at 50 neighboring midquotes. When those neighbors also contain the displaced level, the local mean absolute deviation is large and the center is not dropped. Q3 does not catch the same quotes because the spread is narrow.

After cleaning, several thousand JPM midquotes remain below 23. The largest cleaned JPM five-minute return is still about 11 percent at 10:40, again from a clustered displaced quote rather than an isolated print. Q4 is an isolated-outlier rule. It is not a burst filter. Those remaining quotes are reported, not deleted by an extra ad hoc threshold.


## Cleaned five-minute returns

Large returns are diagnostics. They are not deletion rules.


In [ ]:
sanity = pd.DataFrame(report["return_sanity_cleaned"]["by_stock"]).T
display(sanity)
print("JPM cleaned midquotes below 23 (level diagnostic only)",
      report["n_jpm_cleaned_midquotes_below_23"])

## First cleaned 5×5 realized covariance

Computed with `daily_realized_covariance` on the complete cleaned five-minute panel after dropping the missing 09:30 row. Asset order is IBM, AAPL, MSFT, JPM, XOM. The matrix is an unscaled Gram matrix $R^{\top}R$. No shrinkage or eigenvalue repair is applied.


In [ ]:
assets = report["rcov_cleaned"]["assets"]
rcov = pd.DataFrame(report["rcov_cleaned"]["rcov"], index=assets, columns=assets)
corr = pd.DataFrame(report["rcov_cleaned"]["realized_correlation"], index=assets, columns=assets)
print("Cleaned RCov")
display(rcov)
print("Realized variances")
print(pd.Series(report["rcov_cleaned"]["realized_variance"]))
print("symmetry error", report["rcov_cleaned"]["symmetry_error"])
print("eigenvalues", report["rcov_cleaned"]["eigenvalues"])
print("min eigenvalue", report["rcov_cleaned"]["min_eigenvalue"])
print("numerical rank", report["rcov_cleaned"]["numerical_rank"])
print("PSD", report["rcov_cleaned"]["psd"])
print("condition number", report["rcov_cleaned"]["condition_number"])
print("Realized correlation")
display(corr)

## Naive versus cleaned matrices

This comparison is diagnostic. The cleaned matrix is still a noisy proxy. Remaining JPM artifacts continue to inflate the JPM diagonal relative to IBM, AAPL, MSFT, and XOM, but less than the fully naive panel.


In [ ]:
cmp = report["naive_vs_cleaned"]
print("Frobenius norm of RCov_naive - RCov_clean", cmp["frobenius_norm_difference"])
print("JPM realized variance", cmp["jpm_realized_variance"])
jpm_row = pd.DataFrame(
    {
        "naive": cmp["jpm_covariance_row"]["naive"],
        "cleaned": cmp["jpm_covariance_row"]["cleaned"],
    },
    index=cmp["jpm_covariance_row"]["assets"],
)
print("JPM covariance row")
display(jpm_row)
print(report["caveat"])

## References

Barndorff-Nielsen, O. E., Hansen, P. R., Lunde, A., and Shephard, N. (2009). *Realised kernels in practice: trades and quotes*. The Econometrics Journal, 12(3), C1–C32.

Barndorff-Nielsen, O. E., Hansen, P. R., Lunde, A., and Shephard, N. (2011). *Multivariate realised kernels: consistent positive semi-definite estimators of the covariation of equity prices with noise and non-synchronous trading*. Journal of Econometrics, 162(2), 149–169.
